In [ ]:
!pip install tensorflow scikit-learn matplotlib -q

In [ ]:
# Week 11 Setup — Advanced Neural Networks and Attention Models

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    SimpleRNN,
    LSTM,
    Attention,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

np.random.seed(42)
tf.random.set_seed(42)

print("Google Colab environment is ready")
print("TensorFlow version:", tf.__version__)

In [ ]:
# WEEK 11: ADVANCED NEURAL NETWORKS AND ATTENTION MODELS

print("BIT4133: NATURAL LANGUAGE PROCESSING WITH DEEP LEARNING")
print("Week 11: Advanced Neural Networks and Attention Models")
print("=" * 70)

objectives = [
    "Explain limitations of Recurrent Neural Networks",
    "Describe the Attention Mechanism",
    "Explain how Attention improves sequence learning",
    "Differentiate between RNN, LSTM and Attention models",
    "Implement an Attention mechanism using TensorFlow",
    "Compare an Attention model with a simple RNN model"
]

for i, objective in enumerate(objectives, 1):
    print(f"{i}. {objective}")

In [ ]:
# Why Attention is Needed

print("WHY ATTENTION IS NEEDED")
print("=" * 55)

limitations = [
    "RNNs may forget information in long sentences",
    "Important words may lose influence as sequences become longer",
    "RNNs can suffer from the vanishing gradient problem",
    "Sequential processing may make training slower"
]

for limitation in limitations:
    print("-", limitation)

print()
print("Attention helps a model focus directly on the most relevant words in a sentence.")

In [ ]:
# RNN vs LSTM vs Attention

comparison = [
    ("Handles sequences", "Yes", "Yes", "Yes"),
    ("Long-term memory", "Limited", "Better", "Excellent"),
    ("Focuses on important words", "No", "Limited", "Yes"),
    ("Suitable for long text", "Limited", "Better", "Better"),
    ("Used in modern NLP", "Sometimes", "Yes", "Yes")
]

print(f"{'Feature':<32} {'RNN':<12} {'LSTM':<12} {'Attention'}")
print("-" * 70)

for feature, rnn, lstm, attention in comparison:
    print(f"{feature:<32} {rnn:<12} {lstm:<12} {attention}")

In [ ]:
# Sentiment Analysis Dataset

texts = [
    "the movie was excellent and inspiring",
    "the acting was brilliant and emotional",
    "i enjoyed every moment of the film",
    "the story was engaging and well written",
    "the characters were believable and memorable",
    "the film was entertaining from beginning to end",
    "the director created a powerful story",
    "the ending was satisfying and meaningful",
    "the performance was outstanding",
    "the soundtrack was beautiful",
    "the movie was enjoyable and interesting",
    "the plot was creative and exciting",
    "the actors delivered strong performances",
    "the film exceeded my expectations",
    "the story kept me interested throughout",
    "the movie was touching and thoughtful",
    "the scenes were beautifully presented",
    "the dialogue was natural and effective",
    "the film was worth watching",
    "the experience was positive and enjoyable",
    "the movie was terrible and boring",
    "the acting was weak and unconvincing",
    "i disliked the entire film",
    "the story was confusing and poorly written",
    "the characters were dull and forgettable",
    "the film was disappointing from start to finish",
    "the director created a weak story",
    "the ending was frustrating and meaningless",
    "the performance was extremely poor",
    "the soundtrack was annoying",
    "the movie was unpleasant and uninteresting",
    "the plot was predictable and slow",
    "the actors delivered weak performances",
    "the film failed to meet my expectations",
    "the story lost my interest quickly",
    "the movie was empty and disappointing",
    "the scenes were badly presented",
    "the dialogue was unnatural and ineffective",
    "the film was not worth watching",
    "the experience was negative and unpleasant",
    "the legal assistant explained the ruling clearly",
    "the chatbot gave a helpful and accurate response",
    "the system made legal information easier to understand",
    "the application correctly identified the court entities",
    "the legal summary was clear and informative",
    "the translator produced useful legal translations",
    "the legal assistant gave confusing information",
    "the chatbot produced an inaccurate response",
    "the system made the legal issue harder to understand",
    "the application failed to identify important entities",
    "the legal summary was unclear and misleading",
    "the translator produced incorrect legal translations",
    "the service was fast and reliable",
    "the results were accurate and useful",
    "the application performed very well",
    "the system was slow and unreliable",
    "the results were inaccurate and useless",
    "the application performed very poorly",
    "the response was helpful and professional",
    "the response was rude and unhelpful"
]

labels = [
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0,
    1,1,1,1,1,1,
    0,0,0,0,0,0,
    1,1,1,
    0,0,0,
    1,0
]

print("Dataset size:", len(texts))
print("Positive samples:", labels.count(1))
print("Negative samples:", labels.count(0))

for i in range(5):
    print(texts[i], "->", labels[i])

In [ ]:
# Split the Dataset

X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

In [ ]:
# Tokenization and Vocabulary Building

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)
print("First 20 words in the word index:")

for word, index in list(tokenizer.word_index.items())[:20]:
    print(word, "->", index)

In [ ]:
# Convert Text to Padded Sequences

max_length = 12

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

y_train = np.array(y_train)
y_test = np.array(y_test)

print("Training sequence shape:", X_train_pad.shape)
print("Testing sequence shape:", X_test_pad.shape)
print()
print("Example padded sequence:")
print(X_train_pad[0])

In [ ]:
# Build the Baseline Simple RNN Model

rnn_model = Sequential([
    Embedding(vocab_size, 32),
    SimpleRNN(32),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

In [ ]:
# Train the Baseline RNN Model

rnn_history = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=8,
    verbose=1
)

print("Baseline RNN training completed")

In [ ]:
# Evaluate the Baseline RNN

rnn_loss, rnn_accuracy = rnn_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("BASELINE RNN RESULTS")
print("=" * 55)
print("Test Loss:", round(rnn_loss, 4))
print("Test Accuracy:", round(rnn_accuracy, 4))

In [ ]:
# Build the LSTM with Attention Model

inputs = Input(shape=(max_length,))

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=32
)(inputs)

lstm_output = LSTM(
    64,
    return_sequences=True
)(embedding)

attention_output = Attention()([
    lstm_output,
    lstm_output
])

context_vector = GlobalAveragePooling1D()(attention_output)

dropout = Dropout(0.3)(context_vector)

outputs = Dense(
    1,
    activation="sigmoid"
)(dropout)

attention_model = Model(
    inputs=inputs,
    outputs=outputs
)

attention_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

attention_model.summary()

In [ ]:
# Train the Attention Model

attention_history = attention_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=8,
    verbose=1
)

print("Attention model training completed")

In [ ]:
# Evaluate the Attention Model

attention_loss, attention_accuracy = attention_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("ATTENTION MODEL RESULTS")
print("=" * 55)
print("Test Loss:", round(attention_loss, 4))
print("Test Accuracy:", round(attention_accuracy, 4))

In [ ]:
# Plot RNN Training Accuracy

plt.figure(figsize=(8, 5))
plt.plot(rnn_history.history["accuracy"], label="Training Accuracy")
plt.plot(rnn_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Baseline RNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot Attention Model Training Accuracy

plt.figure(figsize=(8, 5))
plt.plot(attention_history.history["accuracy"], label="Training Accuracy")
plt.plot(attention_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM with Attention Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot Attention Model Training Loss

plt.figure(figsize=(8, 5))
plt.plot(attention_history.history["loss"], label="Training Loss")
plt.plot(attention_history.history["val_loss"], label="Validation Loss")
plt.title("LSTM with Attention Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Compare RNN and Attention Models

print("MODEL PERFORMANCE COMPARISON")
print("=" * 65)
print(f"{'Model':<25} {'Test Accuracy':<20} {'Test Loss'}")
print("-" * 65)
print(f"{'Simple RNN':<25} {rnn_accuracy:<20.4f} {rnn_loss:.4f}")
print(f"{'LSTM + Attention':<25} {attention_accuracy:<20.4f} {attention_loss:.4f}")

if attention_accuracy > rnn_accuracy:
    print("\nThe Attention model achieved higher test accuracy.")
elif attention_accuracy < rnn_accuracy:
    print("\nThe Simple RNN achieved higher test accuracy on this small dataset.")
else:
    print("\nBoth models achieved the same test accuracy.")

In [ ]:
# Classification Report for the Attention Model

attention_probabilities = attention_model.predict(
    X_test_pad,
    verbose=0
)

attention_predictions = (
    attention_probabilities >= 0.5
).astype(int).flatten()

print("ATTENTION MODEL CLASSIFICATION REPORT")
print("=" * 55)
print(
    classification_report(
        y_test,
        attention_predictions,
        target_names=["Negative", "Positive"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, attention_predictions))

In [ ]:
# Sentiment Prediction Function

def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    probability = attention_model.predict(
        padded,
        verbose=0
    )[0][0]

    if probability >= 0.5:
        label = "Positive"
        confidence = probability
    else:
        label = "Negative"
        confidence = 1 - probability

    return label, confidence

print("Sentiment prediction function is ready")

In [ ]:
# Test the Attention Model on New Sentences

test_sentences = [
    "the movie was excellent and enjoyable",
    "the film was boring and disappointing",
    "the legal chatbot gave an accurate answer",
    "the legal assistant was confusing and unhelpful",
    "the system produced useful results"
]

print("ATTENTION-BASED SENTIMENT PREDICTIONS")
print("=" * 65)

for text in test_sentences:
    label, confidence = predict_sentiment(text)
    print("Text:", text)
    print("Prediction:", label)
    print("Confidence:", round(float(confidence), 4))
    print("-" * 65)

In [ ]:
# User Input Sentiment Application

user_text = input("Enter a sentence for sentiment analysis: ").strip()

if user_text:
    label, confidence = predict_sentiment(user_text)

    print("Predicted Sentiment:", label)
    print("Confidence:", round(float(confidence), 4))
else:
    print("No sentence was entered.")

In [ ]:
# Advantages of Attention Mechanisms

print("ADVANTAGES OF ATTENTION")
print("=" * 55)

advantages = [
    "Focuses on the most relevant words",
    "Improves handling of long sequences",
    "Reduces information loss",
    "Supports better contextual understanding",
    "Forms the foundation of Transformer models",
    "Useful in translation, summarization and question answering"
]

for advantage in advantages:
    print("-", advantage)

In [ ]:
# Limitations and Possible Improvements

print("LIMITATIONS")
print("=" * 55)

limitations = [
    "The dataset is small",
    "Training results may vary",
    "The model may overfit",
    "Attention increases model complexity",
    "The dataset contains simple sentences"
]

for limitation in limitations:
    print("-", limitation)

print()
print("POSSIBLE IMPROVEMENTS")
print("=" * 55)

improvements = [
    "Use a larger real-world review dataset",
    "Increase the number of training samples",
    "Tune the LSTM units and embedding size",
    "Use Bidirectional LSTM",
    "Use pre-trained word embeddings",
    "Extend the system using BERT or GPT"
]

for improvement in improvements:
    print("-", improvement)

In [ ]:
# Recommendation for Transformer Extension

print("TRANSFORMER EXTENSION RECOMMENDATION")
print("=" * 55)

print(
    "The system can be improved by replacing the LSTM and Attention layers "
    "with a pre-trained Transformer model such as BERT. BERT already uses "
    "multi-head self-attention and can understand context from both directions. "
    "GPT can also be used for text generation and conversational applications."
)

In [ ]:
# Student Reflection

reflection = (
    "This session helped me understand why Attention mechanisms are important "
    "in sequence learning. I built and compared a Simple RNN with an LSTM "
    "Attention model for sentiment analysis. I learned how Attention helps a "
    "model focus on important words, improves contextual understanding, and "
    "forms the foundation of modern Transformer models."
)

print("STUDENT REFLECTION")
print("=" * 55)
print(reflection)